# 🔍 Eksperimen Machine Learning
## Credit Card Fraud Detection

| | |
|---|---|
| **Nama** | AryaChoirulFikri |
| **Dataset** | Credit Card Fraud Detection (Kaggle) |
| **Tujuan** | Mendeteksi transaksi fraudulent menggunakan ML |
| **Sumber** | https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud |

---

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print('✅ Library berhasil diimport')

## 2. Load Data

In [ ]:
df = pd.read_csv('../creditcard_raw/creditcard.csv')
print(f'Shape  : {df.shape}')
print(f'Kolom  : {df.columns.tolist()}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Distribusi Target (Class)

In [ ]:
class_counts = df['Class'].value_counts()
print('Distribusi kelas:')
print(f'  Normal (0) : {class_counts[0]:,} ({class_counts[0]/len(df)*100:.2f}%)')
print(f'  Fraud  (1) : {class_counts[1]:,} ({class_counts[1]/len(df)*100:.4f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Normal', 'Fraud'], class_counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('Distribusi Kelas (Count)')
axes[0].set_ylabel('Jumlah')

axes[1].pie(class_counts.values, labels=['Normal', 'Fraud'],
            autopct='%1.2f%%', colors=['steelblue', 'crimson'])
axes[1].set_title('Proporsi Kelas')

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('⚠️  Dataset sangat imbalanced — perlu SMOTE!')

### 3.2 Missing Values

In [ ]:
missing = df.isnull().sum()
print(f'Total missing values: {missing.sum()}')
if missing.sum() == 0:
    print('✅ Tidak ada missing values!')
else:
    print(missing[missing > 0])

### 3.3 Distribusi Amount dan Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['Amount'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Amount')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frekuensi')

axes[1].hist(df['Time'], bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Distribusi Time')
axes[1].set_xlabel('Time (detik)')
axes[1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.savefig('eda_amount_time.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Amount: Fraud vs Normal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

normal_amount = df[df['Class'] == 0]['Amount']
fraud_amount  = df[df['Class'] == 1]['Amount']

axes[0].hist(normal_amount, bins=50, color='steelblue', edgecolor='white', alpha=0.7, label='Normal')
axes[0].set_title('Amount — Transaksi Normal')
axes[0].set_xlabel('Amount')

axes[1].hist(fraud_amount, bins=50, color='crimson', edgecolor='white', alpha=0.7, label='Fraud')
axes[1].set_title('Amount — Transaksi Fraud')
axes[1].set_xlabel('Amount')

plt.tight_layout()
plt.savefig('eda_amount_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Rata-rata Amount Normal : Rp {normal_amount.mean():.2f}')
print(f'Rata-rata Amount Fraud  : Rp {fraud_amount.mean():.2f}')

### 3.5 Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
corr_with_target = df.corr()['Class'].drop('Class').sort_values()
corr_with_target.plot(kind='barh', color=['crimson' if v < 0 else 'steelblue' for v in corr_with_target])
plt.title('Korelasi Fitur terhadap Class (Target)')
plt.xlabel('Korelasi')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocessing

### 4.1 Handling Missing Values

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
# Dataset ini tidak ada missing values, tapi kita tetap tulis kodenya
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)
print('✅ Handling missing values selesai')

### 4.2 Handling Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f'Baris dihapus : {before - len(df)}')
print(f'Shape sekarang: {df.shape}')

### 4.3 Feature Engineering

In [ ]:
# Ekstrak jam transaksi
df['Hour'] = (df['Time'] // 3600) % 24

# Log transform Amount
df['Log_Amount'] = np.log1p(df['Amount'])

# Drop kolom asli
df.drop(columns=['Time', 'Amount'], inplace=True)

print(f'Shape setelah feature engineering: {df.shape}')
df[['Hour', 'Log_Amount']].describe()

### 4.4 Feature Scaling

In [ ]:
scaler = StandardScaler()
scale_cols = ['Hour', 'Log_Amount']
df[scale_cols] = scaler.fit_transform(df[scale_cols])
print('✅ StandardScaler diterapkan pada:', scale_cols)
df[scale_cols].describe()

### 4.5 Train-Test Split + SMOTE

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train sebelum SMOTE: {X_train.shape} | fraud={y_train.sum()}')
print(f'Test               : {X_test.shape}  | fraud={y_test.sum()}')

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'Train setelah SMOTE: {X_train_res.shape} | fraud={y_train_res.sum()}')

### 4.6 Simpan Hasil Preprocessing

In [ ]:
import os
os.makedirs('creditcard_preprocessing', exist_ok=True)

train_df = pd.concat([
    pd.DataFrame(X_train_res, columns=X.columns),
    pd.Series(y_train_res, name='Class')
], axis=1)

test_df = pd.concat([
    X_test.reset_index(drop=True),
    y_test.reset_index(drop=True)
], axis=1)

train_df.to_csv('creditcard_preprocessing/train.csv', index=False)
test_df.to_csv('creditcard_preprocessing/test.csv', index=False)
df.to_csv('creditcard_preprocessing/creditcard_preprocessed.csv', index=False)

print('✅ Dataset berhasil disimpan:')
print('   creditcard_preprocessing/train.csv')
print('   creditcard_preprocessing/test.csv')
print('   creditcard_preprocessing/creditcard_preprocessed.csv')

## 5. Kesimpulan

| Tahap | Hasil |
|-------|-------|
| Load Data | 284.807 baris, 31 kolom |
| Missing Values | Tidak ada |
| Duplicates | Dihapus |
| Feature Engineering | Tambah Hour, Log_Amount; drop Time, Amount |
| Scaling | StandardScaler pada Hour & Log_Amount |
| SMOTE | Train set diseimbangkan |
| Output | train.csv, test.csv, creditcard_preprocessed.csv |

Dataset siap digunakan untuk tahap pelatihan model.